# DFTHC compression with fixed angle words

This experiment keeps the verified public three-task portfolio and searches only its FeMoCo component.
**Hypothesis:** joint float64 refitting of `W` and `H2_I`, interleaved with exact column gauges, can escape
restricted amplitude/offset optima; warm-starting C=39 from a compressed C=40 factor can also reduce
state-preparation cost. The angle words, all fifteen loaders, invsqrt and DQI remain byte-identical to the release.
This is an experiment, with no measured improvement claimed before a Kaggle run.

**Kaggle setup:** Accelerator **None / CPU**, Internet **off**, Inputs: competition
`google-verifiable-quantum-circuit-compilation` + **heliosli/qcc-circuits-live**.
No installation, external network access, extra dataset or GPU is needed.

**Safety contract:** first verify the public portfolio with the unmodified official verifiers at default shot
counts and immediately write a valid `/kaggle/working/submission.zip`. Search runs in a terminable child process.
A replacement must pass the full unmodified official DFTHC verifier and have **strictly lower total cost**;
timeouts, exceptions, numerical mismatches and rejected candidates retain the public component.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────────────────
TIME_BUDGET_MINUTES = 240       # total session target; 60..300, including verification reserve
FINAL_RESERVE_MINUTES = 40     # stop search early enough to verify at most two numerical finalists
SEED = 20261102
ERROR_CAP = 0.0199998          # Ha: 19.9998 mHa, positive margin below both official 20 mHa limits
GAP_FLOOR = 3.914900           # strict official floor is 3.914898
LAMBDA_FLOOR = 49.00001        # strict official floor is 49.0
assert 60 <= TIME_BUDGET_MINUTES <= 300
assert FINAL_RESERVE_MINUTES >= 30
# ── Public release and official reference costs ─────────────────────────────────────────────
SALT = 'flexonafft-dfthc-compression-v2'        # local verifier salt; the leaderboard uses its own secret salt
# Either public release works; both ship the same three circuits byte for byte.
#   heliosli/qcc-circuits-live       QCC_RELEASE_MANIFEST.json, files stored as .xz with SHA-256 per file
#   yoyobar/qcc-local-sota-20261031  portfolio-manifest.json, a ready submission/ tree
RELEASES = {'heliosli/qcc-circuits-live': 'QCC_RELEASE_MANIFEST.json',
            'yoyobar/qcc-local-sota-20261031': 'portfolio-manifest.json'}

# Pinned bytes. A mismatch stops the notebook instead of submitting an unknown circuit.
PINNED_SHA256 = {
    'invsqrt.kmb': 'bce2948ff24e20fe59666d131f7fc4316a905103937272c52fdd39d494b6fe47',   # Ehan Mallick qf2
}

# Reference costs of the sample submission (score 1.0), as the paper's verifiers report them.
REFERENCE_COST = {
    'invsqrt': 2_076_774.233715282,          # Q 260 x C 7,987.59 (our run of the official verifier)
    'dqi': 24_439_463_583_672.93,            # Q 1,893, T ~5.725M, R(eps+) 1.1913 (as SomeJoint and busyaprime report)
    'dfthcsos': 3_355_599_588.3155365,       # lambda_eff x T x Q of the provided baseline
}
TASKS = ('invsqrt', 'dqi', 'dfthcsos')

In [ ]:
# ── Session ────────────────────────────────────────────────────────────────────────────────
import os, sys, glob, json, lzma, time, math, shutil, hashlib, zipfile
os.environ.update(OMP_NUM_THREADS="4", OPENBLAS_NUM_THREADS="4", MKL_NUM_THREADS="4")
from pathlib import Path

NB_T0 = time.time()
ROOT = Path('/kaggle/working'); ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT)
INPUT = Path('/kaggle/input')
SUB = ROOT / 'submission'

def deep_find(name: str) -> list[Path]:
    """Recursive search under /kaggle/input that follows symlinked mounts (Path.rglob does not in 3.13)."""
    return sorted(Path(p) for p in glob.glob(str(INPUT / '**' / name), recursive=True))

def inventory(depth: int = 3) -> str:
    """The first levels of /kaggle/input, to show what is attached."""
    lines = []
    for p in sorted(glob.glob(str(INPUT / '*'))) + sorted(glob.glob(str(INPUT / '*' / '*'))) + sorted(glob.glob(str(INPUT / '*' / '*' / '*'))):
        lines.append('  ' + p)
    return 'Attached inputs:\n' + ('\n'.join(lines[:60]) or '  (nothing)')

def find_competition() -> Path:
    for p in (INPUT / 'competitions' / 'google-verifiable-quantum-circuit-compilation',
              INPUT / 'google-verifiable-quantum-circuit-compilation'):
        if (p / 'verifiers').is_dir():
            return p
    hits = deep_find('verify_submission_invsqrt.py')
    if hits:
        return hits[0].parent.parent.parent
    raise FileNotFoundError('Competition data not found: add the competition to the notebook inputs.\n' + inventory())

def find_release() -> tuple[str, Path]:
    for name, marker in RELEASES.items():
        hits = deep_find(marker)
        if len(hits) == 1:
            return name, hits[0].parent
        if len(hits) > 1:
            raise FileNotFoundError(f'{len(hits)} copies of {marker}: attach one version of {name}.\n' + inventory())
    raise FileNotFoundError('No release dataset found. Add one of ' + ', '.join(RELEASES)
                            + ' in Input -> Add Input.\n' + inventory())

COMP = find_competition()
RELEASE_DATASET, RELEASE = find_release()
sys.path[:0] = [str(COMP / 'kickmix'), str(COMP / 'verifiers')]
import numpy as np
import pandas as pd
import kickmix as km
from invsqrt import verify_submission_invsqrt as v_invsqrt
from dqi import verify_submission_dqi as v_dqi
from dfthcsos import verify as v_dfthcsos

print('competition data :', COMP)
print('release dataset  :', RELEASE_DATASET, '->', RELEASE)
print('CPU cores        :', os.cpu_count())
DEADLINE = NB_T0 + 60 * TIME_BUDGET_MINUTES
SEARCH_DEADLINE = DEADLINE - 60 * FINAL_RESERVE_MINUTES

## Restore the authenticated public portfolio

Reuse the session and release extraction from `01_public_portfolio.ipynb`. The glob follows Kaggle's symlinked
input mounts; `.xz` payloads are checked against their release SHA-256. Retain a separate immutable public tree.

In [ ]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

if SUB.exists():
    shutil.rmtree(SUB)
manifest = json.loads((RELEASE / RELEASES[RELEASE_DATASET]).read_text())
if RELEASE_DATASET.startswith('heliosli/'):
    for item in manifest['files']:
        src, dst = RELEASE / item['stored_as'], SUB / item['file']
        dst.parent.mkdir(parents=True, exist_ok=True)
        if item['stored_as'].endswith('.xz'):
            with lzma.open(src) as fi, open(dst, 'wb') as fo:
                shutil.copyfileobj(fi, fo, 1 << 22)
        else:
            shutil.copyfile(src, dst)
        assert sha256(dst) == item['sha256'], f"{item['file']}: differs from the release manifest"
    print('release portfolio:', manifest.get('portfolio'), '| files matched to the manifest:', len(manifest['files']))
else:
    src = RELEASE / 'submission'
    assert src.is_dir(), f'{src} missing: dataset layout changed'
    shutil.copytree(src, SUB)

FILES = sorted(p.relative_to(SUB).as_posix() for p in SUB.rglob('*') if p.is_file())
angles = sorted(f for f in FILES if f.startswith('dfthcsos/angle_k_'))
expected = {'invsqrt.kmb', 'dqi.kmb', 'dfthcsos/submission.npz', *(f'dfthcsos/angle_k_{i}.kmb' for i in range(len(angles)))}
assert set(FILES) == expected, f'unexpected submission tree: {sorted(set(FILES) ^ expected)}'

HASHES = {f: sha256(SUB / f) for f in FILES}
for name, digest in PINNED_SHA256.items():
    assert HASHES[name] == digest, f'{name}: {HASHES[name]} differs from pinned {digest}'
for f in FILES:
    print(f'{HASHES[f][:16]}  {(SUB / f).stat().st_size / 2**20:8.2f} MiB  {f}')
print(len(FILES), 'files,', len(angles), 'angle loaders; pinned hashes match')

print(json.dumps({k: v for k, v in manifest.items() if k != 'files'}, indent=1)[:3000])
PUBLIC = ROOT / 'public_portfolio'
if PUBLIC.exists():
    shutil.rmtree(PUBLIC)
shutil.copytree(SUB, PUBLIC)
PUBLIC_HASHES = dict(HASHES)

## Verify and package the fallback first

No reduced-shot mode is exposed. The public archive is written before optimization starts, so a failed search
cannot remove a usable submission. Public verification itself must succeed; missing/corrupt inputs cannot be
turned into a valid submission by this notebook.

In [ ]:
def atomic_package(results, chosen):
    files = sorted(p.relative_to(SUB).as_posix() for p in SUB.rglob('*') if p.is_file())
    assert files == FILES
    for task in TASKS:
        assert results[task]['status'] == 'PASS', (task, results[task])
    for f in ('invsqrt.kmb', 'dqi.kmb'):
        assert sha256(SUB / f) == PUBLIC_HASHES[f], f
    hashes = {f: sha256(SUB / f) for f in files}
    archive = ROOT / 'submission.zip'
    temporary = ROOT / 'submission.zip.partial'
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for f in files:
            z.write(SUB / f, f)
    with zipfile.ZipFile(temporary) as z:
        assert sorted(z.namelist()) == files
        assert z.testzip() is None
        for f in files:
            assert hashlib.sha256(z.read(f)).hexdigest() == hashes[f]
    os.replace(temporary, archive)
    gm = math.prod(REFERENCE_COST[t] / float(results[t]['score']) for t in TASKS) ** (1 / 3)
    record = dict(experiment='02_dfthcsos_compression', release_dataset=RELEASE_DATASET,
                  release_manifest_sha256=sha256(RELEASE / RELEASES[RELEASE_DATASET]),
                  salt=SALT, default_shots=True, chosen=chosen, sha256=hashes,
                  public_sha256=PUBLIC_HASHES, archive_sha256=sha256(archive),
                  results=results, expected_geometric_mean=gm,
                  elapsed_seconds=time.time() - NB_T0)
    tmp = ROOT / 'provenance.json.partial'
    tmp.write_text(json.dumps(record, indent=2, default=lambda x: x.item() if isinstance(x, np.generic) else str(x)))
    os.replace(tmp, ROOT / 'provenance.json')
    return gm

PUBLIC_RESULTS = {}
t0 = time.time()
circ = km.Circuit.from_file(str(PUBLIC / 'invsqrt.kmb'), format='kmb')
PUBLIC_RESULTS['invsqrt'] = v_invsqrt.verify(circ, v_invsqrt.ProblemSpecInvSqrt.for_n_p(10), salt=SALT)
print('Public invsqrt:', PUBLIC_RESULTS['invsqrt']['status'], round(time.time()-t0), 's', flush=True)
circ = km.Circuit.from_file(str(PUBLIC / 'dqi.kmb'), format='kmb')
PUBLIC_RESULTS['dqi'] = v_dqi.verify(circ, v_dqi.ProblemSpecDQI.benchmark_instance(), salt=SALT)
del circ
print('Public dqi:', PUBLIC_RESULTS['dqi']['status'], round(time.time()-t0), 's', flush=True)
PUBLIC_RESULTS['dfthcsos'] = v_dfthcsos.verify(str(PUBLIC / 'dfthcsos'), salt=SALT)
for task in TASKS:
    assert PUBLIC_RESULTS[task]['status'] == 'PASS', (task, PUBLIC_RESULTS[task])
RESULTS = dict(PUBLIC_RESULTS)
CHOSEN = {task: 'public' for task in TASKS}
GM = atomic_package(RESULTS, CHOSEN)
print('Verified public fallback packaged; expected GM:', GM, flush=True)

## Cost levers and native math

Cost is `lambda_eff × T_total × Q_total`, with
`lambda_eff = sqrt(E_gap * (2*Lambda - E_gap))`. We keep the 8-bit words and J=5 layout: public reports show
Q=428 is determined by inner preparation, so merely narrowing ANGLE circuits need not reduce Q_total.
The shipped loaders already include Lera's tensor/GF(2) sharing, measurement uncompute and Dmitrii Gluzdov's
k=0,14 plan improvements. Public attempts with ten new seeds per circuit did worse. Reimplementing this compiler
without its pipeline dataset would be a large, poorly supported CPU experiment.

Instead we implement **three coupled levers**: unrestricted W/identity-offset refit at fixed words; exact
orthogonal pair gauges of `[W; H2_I]`; and **C=39 rank compression** followed by joint refit. C changes the
number of copies in state preparation, without changing R, B, addresses, words, J or any loader. The verifier
prices the resulting T and Q; no claimed saving substitutes for its total score. We deliberately do not invent
NPZ knobs for state-preparation internals: only actual W/H2_I shapes are changed.

We read and hash `verify.py`, `dfthc_math.py`, `costs_prepare.py`, `circuits.py` and the scoring contract from the
mounted competition at runtime. The fast analytic objective is checked against the verifier's own
`evaluate_dfthc_angle_words` and `evaluate_words` functions, including nonzero perturbations. Those official
functions gate every saved checkpoint. The final acceptance calls `verify` with all default shots.

In [ ]:
import inspect, multiprocessing as mp
from scipy.optimize import minimize, linear_sum_assignment
from dfthcsos.scoreinfra import dfthc_math as official_math, reference
from dfthcsos.scoreinfra.circuits import register_layout

SOURCE_HASHES = {}
for name in ('verify.py', 'dfthc_math.py', 'costs_prepare.py', 'circuits.py', 'SCORING_README.md'):
    hits = sorted((COMP / 'verifiers' / 'dfthcsos').rglob(name))
    for p in hits:
        SOURCE_HASHES[str(p.relative_to(COMP))] = sha256(p)
        text = p.read_text()
        print('\n', p.name, 'sha256', sha256(p))
        if name.endswith('.md'):
            print(text[:3500])
        else:
            print('\n'.join(line for line in text.splitlines() if line.startswith(('def ', 'class ')))[:2500])
with np.load(PUBLIC / 'dfthcsos' / 'submission.npz', allow_pickle=False) as z:
    BASE = {k: np.array(z[k]) for k in z.files}
BROT = register_layout([km.Circuit.from_file(str(PUBLIC / 'dfthcsos' / 'angle_k_0.kmb'), format='kmb')]).brot
INPUTS = v_dfthcsos.load_inputs(PUBLIC / 'dfthcsos' / 'submission.npz')
REF, SHIFT = INPUTS.ref, INPUTS.S
N = int(REF['orbitals'])
VECTORS = official_math.reconstruct_u_vectors(BASE['angle_words'] / 2**(BROT-1), BASE['tree_edges'])
VECTORS /= np.linalg.norm(VECTORS, axis=1, keepdims=True)
BODY = VECTORS[N:].reshape(BASE['W'].shape[0], BASE['W'].shape[1], N)
assert BASE['W'].ndim == 3 and BASE['H2_I'].shape == BASE['W'].shape[::2]
print('NPZ fields:', {k: (v.shape, str(v.dtype)) for k, v in BASE.items()})

def native_metrics(w, p):
    return official_math.evaluate_dfthc_angle_words(
        BASE['angle_words'], w, p, SHIFT, INPUTS.beta_1, REF,
        brot=BROT, tree_edges=BASE['tree_edges'], energy_upper=reference.UPPER_ENERGY)

def eligible(m):
    keys = ('error_1', 'error_2', 'Lambda', 'E_gap', 'lambda_eff')
    return (all(np.isfinite(m[k]) for k in keys) and m['error_1'] <= ERROR_CAP
            and m['error_2'] <= ERROR_CAP and m['Lambda'] >= LAMBDA_FLOOR
            and m['E_gap'] >= GAP_FLOOR and v_dfthcsos.passes_accuracy(m))

BASE_NATIVE = native_metrics(BASE['W'], BASE['H2_I'] - BASE['W'].sum(axis=1))
assert abs(BASE_NATIVE['lambda_eff'] - PUBLIC_RESULTS['dfthcsos']['lambda_eff']) < 1e-7

## Joint refit in packed pair space

For fixed unit vectors, each copy is `X_rc = Σ_b W_rbc u_rb u_rbᵀ`. We represent symmetric orbital pairs once,
with sqrt(multiplicity) weights; the exact two-body error is the Frobenius norm of `Aᵀ A - target`.
The one-body spectrum uses the official one-to-one maximum-overlap assignment. The optimization variables are
all weights plus `p = H2_I - Σ_b W`; identity offsets and W are fitted together. Analytic derivatives allow
L-BFGS steps over roughly 52,000 variables on CPU. An augmented Lagrangian enforces both errors, Lambda and gap.
No smooth surrogate is allowed to authorize a checkpoint: eligibility uses the original native evaluator.

In [ ]:
class FixedWordModel:
    def __init__(self):
        self.ii, self.jj = np.tril_indices(N)
        self.d = np.sqrt(np.where(self.ii == self.jj, 1., 2.))
        self.projectors = BODY[:, :, self.ii] * BODY[:, :, self.jj] * self.d
        target = REF['tei'][self.ii[:, None], self.jj[:, None], self.ii[None, :], self.jj[None, :]].copy()
        diagonal = (self.ii == self.jj).astype(float)
        shifted = SHIFT[self.ii, self.jj]
        target += .5 * (shifted[:, None]*diagonal[None, :] + diagonal[:, None]*shifted[None, :])
        self.target = target * self.d[:, None] * self.d[None, :]
        self.hbase = (REF['oei'] - .5*np.einsum('prrq->pq', REF['tei'])
                      - .5*int(REF['electrons'])*SHIFT + INPUTS.beta_1*np.eye(N))
        self.constant = float(reference.UPPER_ENERGY) - float(REF['ecore']) + int(REF['electrons'])*INPUTS.beta_1

    def pack(self, w, p):
        return np.r_[w.ravel(), p.ravel()]

    def unpack(self, z, shape):
        size = int(np.prod(shape))
        return z[:size].reshape(shape), z[size:].reshape(shape[::2])

    def evaluate(self, z, shape):
        w, p = self.unpack(z, shape)
        a = np.einsum('rbc,rbl->rcl', w, self.projectors, optimize=True)
        flat = a.reshape(-1, len(self.d))
        delta = flat.T @ flat - self.target
        e2sq = float(np.sum(delta*delta))
        packed_h = -np.einsum('rc,rcl->l', p, a) / self.d
        h = self.hbase.copy()
        h[self.ii, self.jj] += packed_h
        h[self.jj, self.ii] = h[self.ii, self.jj]
        eig, vec = np.linalg.eigh(h)
        rows, cols = linear_sum_assignment(-np.abs(VECTORS[:N] @ vec))
        matched = np.empty(N, dtype=int); matched[rows] = cols
        decoded = VECTORS[:N].T @ (eig[matched, None] * VECTORS[:N])
        residual = decoded - h
        e1sq = float(np.sum(residual*residual))
        h2i = p + w.sum(axis=1)
        norm = np.abs(w).sum(axis=1) + np.abs(h2i)
        lam = float(np.abs(eig).sum() + .25*np.sum(norm*norm))
        gap = float(self.constant + np.abs(eig).sum() - eig.sum() + .5*np.sum(p*p))
        discriminant = gap*(2*lam-gap)
        if discriminant <= 0:
            raise ValueError('Left the physical lambda_eff domain')
        eff = math.sqrt(discriminant)
        signh = (vec*np.sign(eig)) @ vec.T
        row_coeff = np.einsum('ij,jk,ik->i', VECTORS[:N], residual, VECTORS[:N], optimize=True)
        eig_coeff = np.empty(N); eig_coeff[matched] = row_coeff
        errorh = 2*((vec*eig_coeff) @ vec.T - residual)

        def through_h(g):
            orbital = np.einsum('rbp,pq,rbq->rb', BODY, g, BODY, optimize=True)
            gw = -orbital[:, :, None]*p[:, None, :]
            gp = -np.einsum('rcl,l->rc', a, g[self.ii, self.jj]*self.d, optimize=True)
            return gw, gp

        lw, lp = through_h(signh)
        lw += .5*norm[:, None, :]*(np.sign(w)+np.sign(h2i)[:, None, :])
        lp += .5*norm*np.sign(h2i)
        gw, gp = through_h(signh-np.eye(N)); gp += p
        e1w, e1p = through_h(errorh)
        ga = (4*flat @ delta).reshape(a.shape)
        e2w = np.einsum('rcl,rbl->rbc', ga, self.projectors, optimize=True)
        gradlam = self.pack(lw, lp)
        gradgap = self.pack(gw, gp)
        gradients = dict(lambda_eff=(gap*gradlam+(lam-gap)*gradgap)/eff,
                         Lambda=gradlam, E_gap=gradgap,
                         error_1_sq=self.pack(e1w, e1p),
                         error_2_sq=self.pack(e2w, np.zeros_like(p)))
        metrics = dict(lambda_eff=eff, Lambda=lam, E_gap=gap,
                       error_1_sq=e1sq, error_2_sq=e2sq,
                       error_1=math.sqrt(e1sq), error_2=math.sqrt(e2sq))
        return metrics, gradients

    def constraints(self, m, g):
        caps = np.array([ERROR_CAP**2, ERROR_CAP**2, GAP_FLOOR, LAMBDA_FLOOR])
        c = np.array([m['error_1_sq']-caps[0], m['error_2_sq']-caps[1],
                      caps[2]-m['E_gap'], caps[3]-m['Lambda']]) / caps
        jac = np.stack([g['error_1_sq']/caps[0], g['error_2_sq']/caps[1],
                        -g['E_gap']/caps[2], -g['Lambda']/caps[3]])
        return c, jac


def check_model(model, w, p, seed=SEED):
    rng = np.random.default_rng(seed)
    z = model.pack(w, p)
    # Nonzero perturbations avoid almost all L1 cusps in the public checkpoint.
    z += rng.normal(size=z.size)*1e-7
    ww, pp = model.unpack(z, w.shape)
    got, grads = model.evaluate(z, w.shape)
    official = native_metrics(ww, pp)
    for k in ('lambda_eff', 'Lambda', 'E_gap', 'error_1', 'error_2'):
        assert abs(got[k]-official[k]) < 2e-8, (k, got[k], official[k])
    # Use coordinates well inside their L1 cells to avoid cancellation from tiny finite steps.
    direction = rng.normal(size=z.size)
    dw, dp = model.unpack(direction, w.shape)
    interior = np.abs(ww.sum(1)+pp) > 1e-3
    dw *= np.abs(ww) > 1e-4
    # At offset cusps keep H2_I constant while still testing coupled W/p derivatives.
    dp[:] = np.where(interior, dp, -dw.sum(axis=1))
    assert np.linalg.norm(direction) > 0
    direction /= np.linalg.norm(direction)
    h = 1e-5
    wp, pp_plus = model.unpack(z+h*direction, w.shape)
    wm, pp_minus = model.unpack(z-h*direction, w.shape)
    assert np.array_equal(np.sign(wp), np.sign(wm))
    assert np.array_equal(np.sign(pp_plus+wp.sum(1)), np.sign(pp_minus+wm.sum(1)))
    plus, _ = model.evaluate(z+h*direction, w.shape)
    minus, _ = model.evaluate(z-h*direction, w.shape)
    for k in ('lambda_eff', 'Lambda', 'E_gap', 'error_1_sq', 'error_2_sq'):
        finite = (plus[k]-minus[k])/(2*h)
        analytic = float(grads[k] @ direction)
        assert abs(finite-analytic) < 2e-4*max(1., abs(analytic)), (k, finite, analytic, h)
    print('Native parity and joint directional derivatives PASS', w.shape, flush=True)

## Exact gauges and rank continuation

A pair rotation of augmented columns leaves `[W; H2_I][W; H2_I]ᵀ` unchanged. We evaluate all zero crossings
and trigonometric stationary points of its piecewise L1 objective, following sigmaborov's exact gauge search.
For C=39, an SVD identifies each group's weakest augmented singular direction; a Householder reflection isolates it
in one column before dropping that column. This preserves more of the existing gauge than rotating every column. Dropping is **approximate**, so the full native errors and official verification remain mandatory.
A failed C=39 attempt does not overwrite a C=40 checkpoint. All shapes keep the public `angle_words` and edges.

In [ ]:
def pair_minimum(x, y):
    radius = np.pi/4
    roots = np.r_[np.arctan2(-x, y), np.arctan2(y, x)]
    roots = (roots+np.pi/2) % np.pi - np.pi/2
    cuts = np.unique(np.r_[-radius, 0., roots[(roots > -radius) & (roots < radius)], radius])
    mid = .5*(cuts[1:]+cuts[:-1])
    c, s = np.cos(mid)[:, None], np.sin(mid)[:, None]
    sx, sy = np.sign(c*x+s*y), np.sign(c*y-s*x)
    p, q, r, u = sx@x, sx@y, sy@y, -sy@x
    aa, bb, cc = p*p+r*r, q*q+u*u, p*q+r*u
    stationary = .5*np.arctan2(-2*cc, bb-aa)
    interior = []
    for shift in (-np.pi/2, 0., np.pi/2):
        points = stationary+shift
        interior.extend(points[(points > cuts[:-1]) & (points < cuts[1:])])
    theta = np.r_[cuts, interior]
    c, s = np.cos(theta)[:, None], np.sin(theta)[:, None]
    values = np.abs(c*x+s*y).sum(1)**2 + np.abs(c*y-s*x).sum(1)**2
    idx = int(np.argmin(values))
    return float(theta[idx]), float(values[idx])


def gauge(w, p, deadline):
    h2i = p+w.sum(axis=1)
    a = np.concatenate([w, h2i[:, None, :]], axis=1)
    original = a.copy()
    rng = np.random.default_rng(SEED)
    pairs = [(r, i, j) for r in range(len(a)) for i in range(a.shape[-1]) for j in range(i+1, a.shape[-1])]
    for index in rng.permutation(len(pairs)):
        if time.time() >= deadline:
            break
        r, i, j = pairs[index]
        x, y = a[r, :, i].copy(), a[r, :, j].copy()
        angle, value = pair_minimum(x, y)
        if value < np.abs(x).sum()**2 + np.abs(y).sum()**2 - 1e-12:
            c, s = np.cos(angle), np.sin(angle)
            a[r, :, i], a[r, :, j] = c*x+s*y, c*y-s*x
    defect = np.max(np.abs(a@a.transpose(0, 2, 1)-original@original.transpose(0, 2, 1)))
    assert defect < 1e-9, defect
    ww = a[:, :-1]
    return ww, a[:, -1]-ww.sum(axis=1)


def drop_weakest_copy(w, p):
    a = np.concatenate([w, (p+w.sum(axis=1))[:, None, :]], axis=1)
    # Augmented Gram is preserved before truncation; no change to loaded orbitals.
    _, _, vh = np.linalg.svd(a, full_matrices=False)
    rotated = []
    for group, weak in zip(a, vh[:, -1]):
        # A Householder aligns only the weakest direction, preserving the other columns' gauge.
        index = int(np.argmax(np.abs(weak)))
        target = np.zeros_like(weak); target[index] = -np.sign(weak[index])
        normal = weak-target; normal /= np.linalg.norm(normal)
        q = np.eye(len(weak))-2*np.outer(normal, normal)
        transformed = group @ q
        assert np.max(np.abs(transformed@transformed.T-group@group.T)) < 1e-9
        rotated.append(np.delete(transformed, index, axis=1))
    rotated = np.stack(rotated)
    ww = rotated[:, :-1]
    return ww, rotated[:, -1]-ww.sum(axis=1)

## Bounded search and atomic checkpoints

Run one native-checked C=40 fit, then a C=39 continuation with independent stages and a final gauge polish.
The child is forcibly stopped at the search deadline. Only eligible checkpoints are saved, by atomic rename;
the parent never relies on a partially written NPZ. Search logs describe numerical results, not leaderboard scores.

In [ ]:
CANDIDATES = ROOT / 'dfthc_candidates'
if CANDIDATES.exists():
    shutil.rmtree(CANDIDATES)
CANDIDATES.mkdir()


def save_native_checkpoint(w, p, label, best):
    native = native_metrics(w, p)
    if not eligible(native) or native['lambda_eff'] >= best.get(label, float('inf')):
        return
    data = dict(BASE, W=w.copy(), H2_I=p+w.sum(axis=1))
    if 'offset_W' in data:
        data['offset_W'] = p.copy()
    assert np.array_equal(data['angle_words'], BASE['angle_words'])
    assert np.array_equal(data['tree_edges'], BASE['tree_edges'])
    tmp = CANDIDATES / (label+'.partial.npz')
    np.savez(tmp, **data)
    # Re-load through the official NPZ schema before making a checkpoint visible.
    check = v_dfthcsos.evaluate_words(v_dfthcsos.load_inputs(tmp), data['angle_words'], data['tree_edges'], BROT)
    assert eligible(check)
    assert abs(check['lambda_eff']-native['lambda_eff']) < 1e-8
    os.replace(tmp, CANDIDATES / (label+'.npz'))
    best[label] = float(native['lambda_eff'])
    print('Native checkpoint', label, {k: float(native[k]) for k in
          ('lambda_eff', 'error_1', 'error_2', 'E_gap', 'Lambda')}, flush=True)


def refit(model, w, p, label, deadline, best):
    shape = w.shape
    z = model.pack(w, p)
    dual = np.zeros(4)
    rho = 10.
    iterations = 0
    best_point = None
    best_value = float('inf')
    last_save = 0

    def checkpoint(point):
        nonlocal iterations, best_point, best_value, last_save
        if time.time() >= deadline:
            raise TimeoutError('Stage deadline')
        iterations += 1
        m, _ = model.evaluate(point, shape)
        if (m['error_1'] <= ERROR_CAP and m['error_2'] <= ERROR_CAP
            and m['E_gap'] >= GAP_FLOOR and m['Lambda'] >= LAMBDA_FLOOR
            and m['lambda_eff'] < best_value):
            ww, pp = model.unpack(point, shape)
            official = native_metrics(ww, pp)
            if eligible(official):
                best_point = point.copy(); best_value = float(official['lambda_eff'])
                if iterations-last_save >= 20:
                    save_native_checkpoint(ww, pp, label, best); last_save = iterations
        if iterations % 25 == 0:
            print(label, 'iteration', iterations, {k: round(m[k], 9) for k in
                  ('lambda_eff', 'error_1', 'error_2', 'E_gap')}, flush=True)

    try:
        for outer in range(8):
            if time.time() >= deadline:
                break
            def objective(point):
                if time.time() >= deadline:
                    raise TimeoutError('Stage deadline')
                m, g = model.evaluate(point, shape)
                c, jac = model.constraints(m, g)
                positive = np.maximum(0., dual+rho*c)
                value = math.log(m['lambda_eff']) + np.sum(positive**2-dual**2)/(2*rho)
                grad = g['lambda_eff']/m['lambda_eff'] + positive @ jac
                return value, grad
            fit = minimize(objective, z, method='L-BFGS-B', jac=True, callback=checkpoint,
                           options=dict(maxiter=250, maxls=30, maxcor=12, ftol=1e-14, gtol=1e-9))
            z = fit.x
            checkpoint(z)
            m, g = model.evaluate(z, shape)
            c, _ = model.constraints(m, g)
            dual = np.maximum(0., dual+rho*c)
            rho = min(rho*10, 1e8)
            print(label, 'outer', outer, str(fit.message), 'constraint', c, flush=True)
    except TimeoutError:
        print(label, 'stage budget exhausted', flush=True)
    if best_point is not None:
        w, p = model.unpack(best_point, shape)
        save_native_checkpoint(w, p, label, best)
    else:
        # An infeasible rank-reduced point is never returned as an accepted checkpoint.
        w, p = model.unpack(z, shape)
    return w.copy(), p.copy()


def search_worker():
    try:
        try:
            from threadpoolctl import threadpool_limits
            threadpool_limits(limits=4)
        except ImportError:
            pass
        model = FixedWordModel()
        w = BASE['W'].copy(); p = BASE['H2_I']-w.sum(axis=1)
        check_model(model, w, p)
        best = {}
        start = time.time()
        available = max(0., SEARCH_DEADLINE-start)
        first_end = start+.40*available
        w, p = gauge(w, p, min(first_end, start+120))
        w, p = refit(model, w, p, 'c40_joint', first_end, best)
        save_native_checkpoint(w, p, 'c40_joint', best)
        # Rank 39 is independent: failure preserves the eligible full-rank checkpoint.
        w39, p39 = drop_weakest_copy(w, p)
        check_model(model, w39, p39, SEED+1)
        rank_end = start+.90*available
        w39, p39 = refit(model, w39, p39, 'c39_joint', rank_end, best)
        if eligible(native_metrics(w39, p39)):
            w39, p39 = gauge(w39, p39, min(SEARCH_DEADLINE, time.time()+120))
            refit(model, w39, p39, 'c39_joint', SEARCH_DEADLINE, best)
        else:
            refit(model, w, p, 'c40_joint', SEARCH_DEADLINE, best)
    except Exception:
        import traceback
        traceback.print_exc()
        print('Search stopped; parent retains the verified public archive.', flush=True)

if time.time() < SEARCH_DEADLINE:
    # Kaggle is Linux; fork keeps the notebook-defined worker callable without writing a script.
    ctx = mp.get_context('fork')
    worker = ctx.Process(target=search_worker)
    worker.start()
    while worker.is_alive() and time.time() < SEARCH_DEADLINE:
        worker.join(timeout=min(30., max(.01, SEARCH_DEADLINE-time.time())))
        print('Search remaining:', round(max(0., SEARCH_DEADLINE-time.time())/60, 1), 'min', flush=True)
    if worker.is_alive():
        worker.terminate(); worker.join(timeout=10)
        if worker.is_alive():
            worker.kill(); worker.join()
    print('Search child exit:', worker.exitcode, flush=True)
else:
    print('Verification used the search allowance; proceeding with public fallback.')

## Official candidate gates

Each saved NPZ is combined with exactly the public loader files and verified at the default shot counts.
Numerical probes are not final verifier results. Search salts and shot counts are never tuned. Full validation
runs in a separate process with a deadline; a timeout or error counts as rejection. The selected archive is
updated only after PASS and a strictly smaller official score.

In [ ]:
def verification_worker(directory, queue):
    try:
        queue.put(v_dfthcsos.verify(str(directory), salt=SALT))
    except Exception as exc:
        queue.put(dict(status='ERROR', reason=repr(exc)))

CANDIDATE_RESULTS = []
for checkpoint in sorted(CANDIDATES.glob('*.npz')):
    if '.partial.' in checkpoint.name:
        continue
    if time.time() >= DEADLINE-90:
        CANDIDATE_RESULTS.append(dict(label=checkpoint.stem, status='TIMEOUT', score=None))
        continue
    candidate = ROOT / ('verify_'+checkpoint.stem)
    try:
        if candidate.exists():
            shutil.rmtree(candidate)
        shutil.copytree(PUBLIC / 'dfthcsos', candidate)
        shutil.copyfile(checkpoint, candidate / 'submission.npz')
        for f in candidate.glob('angle_k_*.kmb'):
            assert sha256(f) == PUBLIC_HASHES['dfthcsos/'+f.name]
        queue = mp.get_context('fork').Queue()
        checker = mp.get_context('fork').Process(target=verification_worker, args=(candidate, queue))
        checker.start()
        result = None
        while time.time() < DEADLINE-60:
            try:
                result = queue.get(timeout=min(15., max(.01, DEADLINE-60-time.time())))
                break
            except Exception as exc:
                from queue import Empty
                if not isinstance(exc, Empty):
                    raise
                if not checker.is_alive():
                    break
        checker.join(timeout=5)
        if checker.is_alive():
            checker.terminate(); checker.join(timeout=5)
            if checker.is_alive():
                checker.kill(); checker.join()
        queue.close()
        result = result or dict(status='TIMEOUT', reason='Full verification did not finish in budget')
        score = float(result['score']) if result.get('status') == 'PASS' else None
        CANDIDATE_RESULTS.append(dict(label=checkpoint.stem, status=result.get('status'), score=score, result=result))
        if score is not None and score < float(RESULTS['dfthcsos']['score']):
            shutil.copyfile(candidate / 'submission.npz', SUB / 'dfthcsos' / 'submission.npz')
            RESULTS['dfthcsos'] = result
            CHOSEN['dfthcsos'] = checkpoint.stem
            GM = atomic_package(RESULTS, CHOSEN)
            print('Accepted full official improvement:', checkpoint.stem, score, flush=True)
        else:
            print('Keep current verified component:', checkpoint.stem, result.get('status'), score, flush=True)
    except Exception as exc:
        CANDIDATE_RESULTS.append(dict(label=checkpoint.stem, status='ERROR', score=None, reason=repr(exc)))
        print('Candidate rejected:', checkpoint.stem, repr(exc), flush=True)

## Candidate versus public and final submission

`ratio` means reference / selected cost. Candidate cost is shown only when a full official verifier returned PASS;
failed or timed-out candidates have no score. Provenance records every submitted file, the archive hash, all
full verifier receipts, and the source hashes of the unmodified verifier modules.

In [ ]:
passed = [r for r in CANDIDATE_RESULTS if r.get('status') == 'PASS' and r.get('score') is not None]
best_candidate = min((r['score'] for r in passed), default=None)
rows = []
for task in TASKS:
    rows.append(dict(task=task, public_cost=float(PUBLIC_RESULTS[task]['score']),
                     candidate_cost=best_candidate if task == 'dfthcsos' else None,
                     ratio=REFERENCE_COST[task]/float(RESULTS[task]['score']), chosen=CHOSEN[task]))
TABLE = pd.DataFrame(rows, columns=['task', 'public_cost', 'candidate_cost', 'ratio', 'chosen'])
print(TABLE.to_string(index=False, na_rep='not accepted'))
for record in CANDIDATE_RESULTS:
    print('Candidate receipt:', record['label'], record['status'], record.get('score'))
for path, digest in SOURCE_HASHES.items():
    assert sha256(COMP / path) == digest, 'Official verifier source changed: '+path
GM = atomic_package(RESULTS, CHOSEN)
provenance_path = ROOT / 'provenance.json'
provenance = json.loads(provenance_path.read_text())
provenance.update(official_source_sha256=SOURCE_HASHES, candidate_results=CANDIDATE_RESULTS,
                  configuration=dict(time_budget_minutes=TIME_BUDGET_MINUTES, error_cap=ERROR_CAP,
                                     gap_floor=GAP_FLOOR, lambda_floor=LAMBDA_FLOOR, seed=SEED),
                  table=rows)
tmp = ROOT / 'provenance.json.partial'
tmp.write_text(json.dumps(provenance, indent=2, default=lambda x: x.item() if isinstance(x, np.generic) else str(x)))
os.replace(tmp, provenance_path)
assert (ROOT / 'submission.zip').is_file()
print('SUBMISSION READY:', ROOT / 'submission.zip', '| expected GM:', f'{GM:.6f}',
      '| official default shots | chosen:', CHOSEN, flush=True)

## Expected effect

Public starting point: lambda_eff ≈47.037, T=16,233, Q=428, cost ≈326,800,030; portfolio GM ≈23.95.
A 0.1% lambda reduction at unchanged resources would give about **0.033%** GM improvement. C=40→39 is a
**2.5% reduction in copy count**, not a 2.5% reduction in total cost: if it saved 90–180 Toffolis in preparation
with Q fixed, chemistry cost would fall about **0.55–1.11%**, giving **0.19–0.37%** GM improvement. These are
conditional estimates; the official cost model determines the actual effect, and rank compression may fail the
20 mHa limits. It could also increase lambda enough to lose all resource savings, which the total-cost gate rejects.

The baseline is already heavily optimized. A further full-rank refit may find nothing and a C=39 fit may remain
infeasible. Reaching GM 28 from 23.95 by chemistry alone would require roughly **37.4% lower chemistry cost**;
this bounded fixed-word study does not promise that. Its contribution is a reproducible test of joint directions
and copy-rank compression with independent native parity/gradient checks and a verified fallback.

# Sources and credits

All angle loader circuits, invsqrt and DQI are public circuits used byte for byte from an Apache-2.0 dataset.
The chemistry weights may be replaced only by this experiment after full verification and a lower-cost result. The lineages below follow the credits the authors themselves give; if anyone is missing, please tell us in the
comments and we will add them.

- **Release datasets:** heliosli, [qcc-circuits-live](https://www.kaggle.com/datasets/heliosli/qcc-circuits-live), from
  [The dataset is public now. Enjoy!](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy), whose
  assembly and re-verification this notebook follows; and yoyobar,
  [qcc-local-sota-20261031](https://www.kaggle.com/datasets/yoyobar/qcc-local-sota-20261031), from
  [LB 22.99 | invsqrt 101 - 62 qubits](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits).

- **invsqrt** — Ehan Mallick, [invsqrt 20.65: two 3-to-2 AND merges on y6225](https://www.kaggle.com/code/ehanmallick/invsqrt-20-65-two-3-to-2-and-merges-on-y6225)
  (two GF(2) quadratic-form windows where three ANDs become two), on yoyobar and GPT-6 Astra's 62-qubit circuit.
  Earlier lineage as the authors credit it: mrmeows / team selenium
  ([21.0 circuits](https://www.kaggle.com/code/mrmeows/21-0-circuits-let-s-all-reach-the-plateau)) → Akmal Xodarev
  (busyaprime) 63 qubits → yoyobar's cone rewrites → Ahmed Berat Özer's phase correction (as distributed in manwithacat's
  [VQCC Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo)); earlier still Phuc Thai (DePie),
  Dmitrii Gluzdov, heliosli and marsyas.

- **dqi** — yoyobar's selected combination of lead-product reuse and product caching on Akmal Xodarev's (busyaprime)
  [Exact DQI Rewrites at 892 Qubits](https://www.kaggle.com/code/busyaprime/lb-23-91003-exact-dqi-rewrites-at-892-qubits)
  (flag clears by X-basis measurement, symbolic AND caching, interval allocation), which uses manwithacat's two-wire
  clean ladder ([Hedge Wizardry HW-07](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook)) and
  Ahmed Berat Özer's product-caching idea, on SomeJoint's (Agniv Banerjee's team)
  [892-qubit decoder](https://www.kaggle.com/code/somejoint/quantum-circuits-23-85-local-gm-892-qubit-dqi). SomeJoint
  credits GPT-6 Astra Pro, Claude, Ali Haydar Özdağ (flag carry), yoyobar (one-carry history) and the earlier decoder
  line: jday96314 (QuantumAI), Alex Chilton, mrmeows, manwithacat, Dmitrii Gluzdov, ViberTeam and Phuc Thai (DePie).

- **dfthcsos** — Lera (marculera), [DF-THC 10x: Compact Groups From the Hamiltonian](https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian):
  the (9, 143, 40) factorization, its 8-bit angle words and the loaders k = 1 to 13; Dmitrii Gluzdov,
  [Quantum Circuit Compilation from Source](https://www.kaggle.com/code/dmitriigluzdov/quantum-circuit-compilation-from-source):
  the loaders k = 0 and k = 14; heliosli: the table weights `W`, `H2_I`, continuing Ahmed Berat Özer's refinement in
  [DF-THC Group Compression](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression); assembly by
  manwithacat ([The Real $150K Was the Friends We Made](https://www.kaggle.com/code/manwithacat/the-real-150k-was-the-friends-we-made));
  Atakan Aldemir pointed heliosli to Lera's tables. Lera's own sources: Ahmed Berat Özer, somejoint, mrmeows, heliosli,
  lightningv08, kozykappa, Kozykorpesh and Sumit Chongder, Artem1410 (anazemcev), haideptry via phucthaiv02,
  Michael Rustad, k256net and Dmitrii Gluzdov.

- **Organizers:** Google Quantum AI (Tanuj Khattar, Nicholas C. Rubin, Craig Gidney, Michael Broughton, Guang Hao Low,
  Ryan Babbush) for the tasks, the verifiers and kickmix
  ([arXiv:2610.23832](https://arxiv.org/abs/2610.23832)), which keep their own terms.

- **Additional numerical methods:** sigmaborov,
  [22.85569 LB | Exact Gauge Search](https://www.kaggle.com/code/sigmaborov/22-85569-lb-exact-gauge-search):
  exact pairwise orthogonal descent of augmented DFTHC coefficient columns; native-math API use and parity/gradient
  checks. Its source states: “Retains A A.T for A=[W; H2_I]. The piecewise trigonometric L1 objective
  is minimized over all zero crossings and all interior stationary points.” The pair solver here follows that
  implementation. sigmaborov credits **avazaku** (Apache-2.0) for the spectral identity-offset and column models;
  the derivation of the one-body derivatives here follows that model. Dmitrii Gluzdov's source explains the
  augmented column gauge and multi-column continuation after **kirilltym**'s
  [Exact Gauge + 5-Word QROM](https://www.kaggle.com/code/kirilltym/quantum-circuits-exact-gauge-5-word-qrom)
  and earlier gauge studies by kozykappa, Phuc Thai, anazemcev, Alex Chilton and avazaku.
- **Constrained-refit idea:** James Barlow (**manwithacat**),
  [Hedge Wizardry Notebook](https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook), whose public
  lineage states: “the fixed-word constrained W/H2_I refit by manwithacat (James Barlow), assisted by Codex
  (OpenAI). The refit method was developed in HW-03 and applied to C42 in HW-04.”
- **Ours:** the joint packed-pair analytic W/identity-offset optimizer, its augmented-Lagrangian schedule,
  SVD-initialized C=39 refit experiment, process deadlines, atomic native checkpoints and full-cost fallback
  integration. SVD truncation and augmented Lagrangians are standard methods; we do not claim to invent them.
  No public circuit is claimed as ours, and no gain is claimed before execution.

The circuits keep their authors' licences; the notebook code is Apache-2.0.